In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import random
import math
import matplotlib
import seaborn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torch.nn.functional as F


In [2]:
# =========================
# SIMPLEX 
# =========================

def simplex(o, c, a, rhs):
    n, m = a.shape
    if o == 'M':
        c = -c

    num_artificials = sum(1 for i in range(n) if rhs[i] < 0)
    total_vars = m + n + num_artificials
    tableau = np.zeros((n + 1, total_vars + 1))

    tableau[:n, :m] = a
    tableau[:n, m:m+n] = np.eye(n)
    tableau[:n, -1] = rhs

    M = 1e7
    basis = list(range(m, m + n))

    art_col = m + n
    for i in range(n):
        if tableau[i, -1] < 0:
            tableau[i, :m+n] *= -1
            tableau[i, -1] *= -1
            tableau[i, art_col] = 1
            basis[i] = art_col
            tableau[n, art_col] = M
            art_col += 1

    for i in range(n):
        if basis[i] >= m + n:
            tableau[n, :] -= M * tableau[i, :]

    tableau[n, :m] += c

    while True:
        neg_cols = np.where(tableau[n, :-1] < -1e-9)[0]
        if len(neg_cols) == 0:
            break

        pivot_col = neg_cols[0]
        ratios = []

        for i in range(n):
            if tableau[i, pivot_col] > 1e-9:
                ratios.append(tableau[i, -1] / tableau[i, pivot_col])
            else:
                ratios.append(np.inf)

        if all(r == np.inf for r in ratios):
            return None

        pivot_row = np.argmin(ratios)
        basis[pivot_row] = pivot_col

        tableau[pivot_row] /= tableau[pivot_row, pivot_col]
        for i in range(n + 1):
            if i != pivot_row:
                tableau[i] -= tableau[i, pivot_col] * tableau[pivot_row]

    for i, b_var in enumerate(basis):
        if b_var >= m + n and tableau[i, -1] > 1e-9:
            return None

    X = np.zeros(m)
    for j in range(m):
        col = tableau[:n, j]
        if np.sum(np.isclose(col, 1)) == 1 and np.sum(np.isclose(col, 0)) == n - 1:
            X[j] = tableau[np.argmax(col), -1]

    return X

In [3]:
#show LPP#

def show(o, c, a, b):
    
    terms = [f"+ {c[i]}x{i+1}" if c[i] >= 0 else f"- {abs(c[i])}x{i+1}" for i in range(len(c))]
    C = " ".join(terms).lstrip("+ ")
    
    if o == 'm':
        print(f"min z = {C}")
        
    else:
        print(f"max z = {C}")
        
    print("\nSubject to\n")
    
    for i in range(len(b)):
        terms = [f"+ {a[i,j]}x{j+1}" if a[i,j] >= 0 else f"- {abs(a[i,j])}x{j+1}" for j in range(len(c))]
        C = " ".join(terms).lstrip("+ ")
        print(f"{C} <= {b[i]}")

In [4]:
#input#

o = input("M/m")

if o not in {"M","m"}:
    raise ValueError(f"Invalid choice '{o}'. Must be 'M' or 'm'.")

m=int(input("number of variables"))

if m > 20:
    raise ValueError(f"number of variables must be {20} or less. You entered {m}.")

c = np.zeros(m)

for i in range (0,m):
    c[i] = float(input(f"c_{i} = "))

n = int(input("number of constraints"))

if n > 30:
    raise ValueError(f"number of constraints must be {30} or less. You entered {n}.")


a = np.zeros((n,m))
b = np.zeros(n)

for i in range (0,n):
    for j in range (0,m):
        a[i,j] = float(input(f"a_{i},{j} = "))
    b[i] = float(input(f"b_{i} = "))
    

M/m m
number of variables 1
c_0 =  1
number of constraints 1
a_0,0 =  1
b_0 =  1


In [5]:
#print LPP#
show(o,c,a,b)

min z = 1.0x1

Subject to

1.0x1 <= 1.0


In [6]:
#solve LPP
X = np.zeros(m)
X = simplex(o, c, a, b)
z = X @ c
print(f"optimal solution is {X}")
print(f"optimal value is {z}")

optimal solution is [1.]
optimal value is 1.0
